# Bedrock ModelBuilder: Deploy Models from S3 Paths

This notebook shows how to deploy fine-tuned models to Amazon Bedrock using `BedrockModelBuilder`.

Three deployment patterns are covered:
1. **From a TrainingJob** — fetch a completed training job and deploy
2. **From a direct S3 URI** — deploy model artifacts at a known S3 path (Nova custom model)
3. **From a direct S3 URI** — deploy model artifacts via OSS model import

In [ ]:
import boto3
import json
import time
from sagemaker.core.resources import TrainingJob
from sagemaker.serve.bedrock_model_builder import BedrockModelBuilder

In [ ]:
# Configuration — update these for your account
REGION = "us-east-1"
ROLE_ARN = "arn:aws:iam::<ACCOUNT_ID>:role/<YOUR_ROLE>"

## Option 1: Deploy from a TrainingJob

Fetch a completed training job and pass it directly. The builder resolves the
model artifacts automatically from `model_artifacts.s3_model_artifacts`.

In [ ]:
TRAINING_JOB_NAME = "my-sft-nova-lite-job"

training_job = TrainingJob.get(
    training_job_name=TRAINING_JOB_NAME,
    region=REGION,
)

print(f"Status: {training_job.training_job_status}")
print(f"Model artifacts: {training_job.model_artifacts.s3_model_artifacts}")

In [ ]:
builder = BedrockModelBuilder(model=training_job)

result = builder.deploy(
    custom_model_name="my-nova-model",
    role_arn=ROLE_ARN,
)

print(f"Deployment ARN: {result.get('customModelDeploymentArn')}")

## Option 2: Deploy from a direct S3 URI (Nova custom model)

If you already know the S3 path to your model checkpoint, pass it directly as a string.
Provide `custom_model_name` to use the Nova `create_custom_model` + deployment path.

In [ ]:
S3_MODEL_PATH = "s3://customer-escrow-123456789012-smtj-abc123/my-job/1920/"

builder = BedrockModelBuilder(model=S3_MODEL_PATH)

result = builder.deploy(
    custom_model_name="my-nova-from-s3",
    role_arn=ROLE_ARN,
)

print(f"Deployment ARN: {result.get('customModelDeploymentArn')}")

## Option 3: Deploy from a direct S3 URI (OSS model import)

For non-Nova models (e.g., Llama, Mistral), omit `custom_model_name`.
The builder uses `create_model_import_job` and polls until the import completes.

In [ ]:
S3_MODEL_PATH = "s3://my-bucket/fine-tuned-llama/hf_merged/"

builder = BedrockModelBuilder(model=S3_MODEL_PATH)

result = builder.deploy(
    job_name="my-llama-import",
    imported_model_name="my-llama-imported",
    role_arn=ROLE_ARN,
)

print(f"Import status: {result.get('status')}")
print(f"Model ARN: {result.get('importedModelArn')}")

## Provisioned Throughput (optional)

After deployment, you can optionally create provisioned throughput for dedicated capacity.
The `model_id` is automatically inferred from the previous `deploy()` call.

In [ ]:
# pt_result = builder.create_provisioned_throughput(
#     provisioned_model_name="my-provisioned-model",
#     model_units=1,
# )

## Test Inference

In [ ]:
# Replace with the model ARN from the deploy result
MODEL_ARN = result.get("customModelDeploymentArn") or result.get("importedModelArn")

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ARN,
    body=json.dumps({
        "messages": [
            {"role": "user", "content": "What is the capital of France?"}
        ],
        "max_tokens": 100,
        "temperature": 0.7,
    }),
)

print(json.loads(response["body"].read().decode()))